[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/spinors/spinors.ipynb)

# Dirac, Weyl and Majorana Spinors

This example computes how a spin-one-half state turns under spatial rotations and boosts, and how handedness and particle-antiparticle symmetry single out Weyl and Majorana states within the same space. Rotating a spinor by one full turn brings its spin axis back to where it started while reversing the sign of the state itself, so interference against an unturned reference switches completely from the sum port to the difference port until a second full turn restores both. Within that eight-dimensional real spinor space, Weyl states carry a definite handedness with lightlike currents along or against the spin axis, whereas Majorana states stay unchanged under charge conjugation.

A spinor is modelled as an even multivector `Spinor` of the spacetime algebra, and its two-sided sandwich `psi >> time` carries the observer's time direction into the future-directed Dirac current while `t & (psi >> time)` reads the probability density on the observer's spatial slice. Phase rotations, chirality, and charge conjugation act on the right as linear maps `Spinor * yx`, `Spinor * zt`, and `Spinor * yt` on that same real space, commuting with every Lorentz rotor acting on the left. Because chirality and charge conjugation each square to the identity map, half-sums such as `(Spinor + chirality_map) / 2` and `(Spinor + charge_conjugation) / 2` project general Dirac states onto four-dimensional Weyl and Majorana subspaces, following [Francis and Kosowsky (2005)](https://arxiv.org/pdf/math-ph/0403040).

In Dirac-matrix notation, a spinor is written as a four-component complex column $\psi \in \mathbb C^4$ acted on by $4 \times 4$ complex matrices $\gamma^\mu$ satisfying $\{\gamma^\mu, \gamma^\nu\} = 2\eta^{\mu\nu} I_4$, with Dirac adjoint $\bar\psi = \psi^\dagger \gamma^0$ and probability current $J^\mu = \bar\psi \gamma^\mu \psi$. Splitting $\psi$ into Weyl parts uses the fifth matrix $\gamma^5 = i\gamma^0\gamma^1\gamma^2\gamma^3$ in $\tfrac12(I_4 \pm \gamma^5)\psi$, while charge conjugation leaves complex-linear matrix multiplication altogether for the antiunitary map $\psi_c = C\bar\psi^{\mathsf T} = i\gamma^2\psi^*$ that conjugates every complex entry before mixing components.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.relativity.spinors import render, scenarios
from examples.util.animation import save_figure, show_states

# Spacetime gives spatial axes square +1 and the time slice t square -1.
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                     # Scalar+
# Spacetime currents and spin axes are intrinsic directions (antivectors).
Antivector = ga.gatype.antivector()                             # Antivector-
Spinor = ga.gatype.even()                                       # Spinor+
# Negating the dual of `mv.t` compensates for `t` squaring to -1 so `mv.t & time == 1`.
time = -mv.t.dual()                                             # [] Antivector-

## Two full turns of a spinor

Half-angle exponentiation of a spatial bivector generates a rotor whose left multiplication turns a spinor while its two-sided sandwich `psi >> direction` turns observable directions by the full angle. Recombining the turned state coherently with an unturned reference splits the observer density across a sum port and a difference port, turning the overall sign of `psi` into an observable shift of intensity.

In [ ]:
n_rotation_frames = scenarios.n_rotation_frames
frame_duration_ms = scenarios.frame_duration_ms
rotation_angles = scenarios.rotation_angles

In [ ]:
def current(psi: Spinor) -> Antivector:
    """The two-sided sandwich of `psi` carries the observer's time direction into the future-directed Dirac current (`[...] Antivector-`)."""
    return psi >> time


def density(psi: Spinor) -> Scalar:
    """Pairing the observer's spatial slice with the Dirac current gives the measured probability density (`[...] Scalar+`)."""
    return mv.t & current(psi)

In [ ]:
def spin(psi: Spinor) -> Antivector:
    """The two-sided sandwich of `psi` carries the reference z direction into the spatial spin current (`[...] Antivector-`)."""
    return psi >> mv.z.dual()


def rotation_states(angles: np.ndarray) -> Iterator[tuple[Antivector, Scalar, Scalar]]:
    """Yields the spatial spin current and the sum-port and difference-port densities at each rotation angle."""
    reference = mv.scalar()                                     # [] Spinor+
    for angle in angles:
        psi = (mv.xz * (angle / 2)).exp()                       # [] Spinor+
        sum_density = density((psi + reference) / 2)            # [] Scalar+
        difference_density = density((psi - reference) / 2)     # [] Scalar+
        yield spin(psi), sum_density, difference_density


rotation_history = show_states(
    rotation_states(rotation_angles),
    render.rotation_animation,
    name="spinor_rotation",
    duration_ms=frame_duration_ms,
    n_frames=n_rotation_frames,
);

In the left panel, the slate arrow traces the spatial spin direction around the x–z unit circle twice. After the first full turn, the arrow points along positive z again while the blue glow moves completely from the sum port to the difference port because the spinor has reversed sign, returning to the sum port only after the second full turn.

## Handedness and Weyl currents

Right multiplication `Spinor * zt` commutes with every left Lorentz rotor and squares to the identity map on `Spinor`. Its two complementary projectors split an eight-component Dirac spinor into four-dimensional left-handed and right-handed Weyl spinors within the same even subalgebra, each carrying a future-directed lightlike current whose sum is the timelike Dirac current and whose difference is the spatial spin current.

In Dirac-matrix notation, the chiral projectors read as $\tfrac12(I_4 \pm \gamma^5)\psi$, and the decomposition of the probability current into future-null chiral pieces reads as $\bar\psi\gamma^\mu\psi = \bar\psi_L\gamma^\mu\psi_L + \bar\psi_R\gamma^\mu\psi_R$.

In [ ]:
rapidity = scenarios.rapidity

In [ ]:
chirality_map = Spinor * mv.zt                                  # [] Spinor+ <- Spinor+
right_projector = (Spinor + chirality_map) / 2                  # [] Spinor+ <- Spinor+
left_projector = (Spinor - chirality_map) / 2                   # [] Spinor+ <- Spinor+


def chiral_currents(rapidity: float) -> tuple[Antivector, Antivector, Antivector]:
    """Projecting a z-boosted Dirac spinor onto left and right Weyl subspaces splits its current into two null currents (`[] Antivector-`)."""
    psi = (mv.tz * (-rapidity / 2)).exp()                       # [] Spinor+
    left_spinor = left_projector(psi)                           # [] Spinor+
    right_spinor = right_projector(psi)                         # [] Spinor+
    return current(left_spinor), current(right_spinor), current(psi)


left_current, right_current, dirac_current = chiral_currents(rapidity)
save_figure(render.chirality_figure(left_current, right_current, dirac_current), "spinor_chirality");

The shaded wedge marks the future light cone in the z–t plane. Boosting along positive z enhances the blue right-handed null current along the forward light ray and suppresses the amber left-handed null current along the backward ray, while their vector sum forms the timelike slate Dirac current inside the cone.

## Charge conjugation and Majorana spinors

Having real components in the spacetime algebra does not by itself make a spinor its own antiparticle, because electromagnetic phase turns also act within the real even subalgebra as right multiplication `Spinor * yx`. Charge conjugation acts as right multiplication `Spinor * yt`, which squares to the identity map and anticommutes with `Spinor * yx` while exchanging left and right Weyl subspaces. Whereas every phase turn leaves a Weyl subspace invariant, a phase turn rotates a Majorana spinor between the `+1` and `-1` eigenspaces of charge conjugation.

In Dirac-matrix notation, the Majorana condition reads as $\psi = \psi_c$ with $\psi_c = C\bar\psi^{\mathsf T} = i\gamma^2\psi^*$, where complex conjugation reverses a phase factor $e^{-i\phi}\psi$ into $e^{+i\phi}\psi_c$.

In [ ]:
n_phase_samples = scenarios.n_phase_samples
phase_angles = scenarios.phase_angles

In [ ]:
phase_unit = Spinor * mv.yx                                     # [] Spinor+ <- Spinor+
charge_conjugation = Spinor * mv.yt                             # [] Spinor+ <- Spinor+
majorana_projector = (Spinor + charge_conjugation) / 2          # [] Spinor+ <- Spinor+


def majorana_densities(phase_angles: np.ndarray) -> tuple[Scalar, Scalar]:
    """Advancing a unit-density Majorana spinor through `phase_angles` shifts observer density between the two conjugation eigenspaces (`[...] Scalar+`)."""
    seed_spinor = majorana_projector(mv.scalar())               # [] Spinor+
    majorana_spinor = seed_spinor / density(seed_spinor).square_root()  # [] Spinor+
    phase_rotors = (mv.yx * phase_angles).exp()                 # [...] Spinor+
    phased_spinors = majorana_spinor * phase_rotors             # [...] Spinor+
    conjugate_spinors = charge_conjugation(phased_spinors)      # [...] Spinor+
    fixed_parts = (phased_spinors + conjugate_spinors) / 2      # [...] Spinor+
    negated_parts = (phased_spinors - conjugate_spinors) / 2    # [...] Spinor+
    return density(fixed_parts), density(negated_parts)


fixed_density, negated_density = majorana_densities(phase_angles)  # [n_phase_samples] Scalar+, Scalar+
save_figure(render.majorana_figure(phase_angles, fixed_density, negated_density), "spinor_majorana");

Starting from a Majorana state at zero phase angle, advancing the phase trades observer density smoothly from the blue `+1` conjugation eigenspace into the amber `-1` eigenspace and back while keeping their sum equal to one. Only phase angles `0`, `π`, and `2π`, corresponding to multiplying the state by `+1` or `-1`, leave the state entirely within the Majorana subspace.

In [ ]:
# checks
initial_spin, initial_sum, initial_difference = rotation_history[0]
half_turn_spin, half_turn_sum, half_turn_difference = rotation_history[n_rotation_frames // 2]
assert np.abs((half_turn_spin - initial_spin).kernel).max() < 1e-9
assert (initial_sum - 1).abs() < 1e-9 and initial_difference.abs() < 1e-9
assert half_turn_sum.abs() < 1e-9 and (half_turn_difference - 1).abs() < 1e-9

assert left_current.squared().abs() < 1e-9
assert right_current.squared().abs() < 1e-9
assert dirac_current.squared() < 0
assert np.abs((left_current + right_current - dirac_current).kernel).max() < 1e-9

assert (fixed_density + negated_density - 1).abs().max() < 1e-9
assert (fixed_density[0] - 1).abs() < 1e-9
assert (negated_density[(n_phase_samples - 1) // 4] - 1).abs() < 1e-9
assert (fixed_density[(n_phase_samples - 1) // 2] - 1).abs() < 1e-9